# Step 2 — Load the data and check that it fits together

This notebook does not search for biomarkers yet. It opens the four files and answers six questions:

1. How big is each table?
2. Which columns will we use later?
3. Which gene-expression rows are the default profile for each cell line?
4. Do the drug codes in the sensitivity table match names in the conditions file?
5. Is venetoclax in the screen?
6. How many cell lines have both expression and a drug measurement?

A **cell line** is cancer cells grown in a dish. DepMap gives every cell line an ID such as `ACH-000001`. That ID is called `ModelID`, and it is the label we will use to line the files up.

## 1. Find the data folder

The notebook lives in `notebooks/`, and the files live in `data/`. Jupyter sometimes treats the project folder as the current folder, and sometimes treats `notebooks/` as the current folder. The next cell checks both.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

here = Path.cwd()
if (here / "data").exists():
    data_dir = here / "data"
elif (here.parent / "data").exists():
    data_dir = here.parent / "data"
else:
    raise FileNotFoundError(
        "Could not find the data folder. Open a terminal, cd into "
        "drug-sensitivity-ai, and run this notebook from there."
    )

print("Reading files from:", data_dir.resolve())
for path in sorted(data_dir.glob("*.csv")):
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"  {path.name}: {size_mb:.1f} MB")

Reading files from: /Users/ellierobinson/drug-sensitivity-ai/data
  GDSC2AUCMatrix.csv: 3.7 MB
  GDSC2Log2ViabilityConditions.csv: 5.6 MB
  Model.csv: 0.8 MB
  OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv: 376.7 MB


## 2. Cell-line catalog: `Model.csv`

This is the phone book. One row is one cell line. It holds the cancer type and the name used in papers. The drug and expression files only know the `ModelID`, so we will use this table whenever we want a readable name or a cancer type.

`OncotreeLineage` is the broad cancer type, such as lung, breast, or lymphoid.

In [2]:
models = pd.read_csv(data_dir / "Model.csv")

print(f"Cell-line catalog: {models.shape[0]} rows and {models.shape[1]} columns")
print(f"Unique ModelIDs: {models['ModelID'].nunique()}")
print()
print("Columns we will use:")
print("  ModelID                 DepMap cell-line ID")
print("  CellLineName            name of the cell line")
print("  OncotreeLineage         broad cancer type")
print("  OncotreePrimaryDisease  more specific disease")
print("  OncotreeSubtype         subtype, when one is listed")
print()
print("Every column in the file:")
print(", ".join(models.columns))

models[["ModelID", "CellLineName", "OncotreeLineage", "OncotreePrimaryDisease", "OncotreeSubtype"]].head()

Cell-line catalog: 2554 rows and 48 columns
Unique ModelIDs: 2554

Columns we will use:
  ModelID                 DepMap cell-line ID
  CellLineName            name of the cell line
  OncotreeLineage         broad cancer type
  OncotreePrimaryDisease  more specific disease
  OncotreeSubtype         subtype, when one is listed

Every column in the file:
ModelID, PatientID, CellLineName, StrippedCellLineName, DepmapModelType, OncotreeLineage, OncotreePrimaryDisease, OncotreeSubtype, OncotreeCode, RRID, Age, AgeCategory, Sex, PatientRace, PrimaryOrMetastasis, SourceType, SourceDetail, CatalogNumber, SampleCollectionSite, ModelSubtypeFeatures, PatientSubtypeFeatures, TissueOrigin, ModelDerivationMaterial, ModelTreatment, PatientTreatmentStatus, PatientTreatmentType, PatientTreatmentDetails, Stage, StagingSystem, PatientTumorGrade, PatientTreatmentResponse, GrowthPattern, OnboardedMedia, FormulationID, SerumFreeMedia, PlateCoating, EngineeredModel, EngineeredModelDetails, CulturedResistance

,ModelID,CellLineName,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype
0,ACH-000001,NIH:OVCAR-3,Ovary/Fallopian Tube,Ovarian Epithelial Tumor,High-Grade Serous Ovarian Cancer
1,ACH-000002,HL-60,Myeloid,Acute Myeloid Leukemia,AML with Myelodysplasia-Related Changes
2,ACH-000003,CACO2,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma
3,ACH-000004,HEL,Myeloid,Acute Myeloid Leukemia,"AML, NOS"
4,ACH-000005,HEL 92.1.7,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia


## 3. Gene expression

`OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv` is the large file. Each row is one RNA-sequencing profile. Each gene column is `log2(TPM + 1)`: TPM means transcripts per million, a standard way to say how much RNA that gene produced. Adding 1 and taking log2 keeps zeros well behaved and makes the numbers easier to compare. A higher number means the gene was expressed more strongly.

The first columns are bookkeeping. The rest, about nineteen thousand of them, are genes. A gene column looks like `TSPAN6 (7105)`: the gene symbol, then its Entrez ID in parentheses.

This file can take about a minute to load.

In [3]:
expression_path = data_dir / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"
print("Loading gene expression. This cell can take about a minute.")

expression_all = pd.read_csv(expression_path)
print(f"Expression file: {expression_all.shape[0]} rows and {expression_all.shape[1]} columns")
print()
print("First six columns:")
print(list(expression_all.columns[:6]))

Loading gene expression. This cell can take about a minute.
Expression file: 2314 rows and 19224 columns

First six columns:
['Unnamed: 0', 'SequencingID', 'ModelConditionID', 'ModelID', 'IsDefaultEntryForMC', 'IsDefaultEntryForModel']


## 4. Keep one default profile per cell line

Some cell lines were sequenced more than once. For a drug comparison we want one expression profile per cell line, the profile DepMap marks as the default.

In this download the flag is `IsDefaultEntryForModel`. Rows to keep are marked `Yes`. (`IsDefaultEntryForMC` is a different flag, for a growth condition. We leave that one alone.)

In [4]:
print("Values in IsDefaultEntryForModel:")
print(expression_all["IsDefaultEntryForModel"].value_counts(dropna=False))
print()
print("Values in IsDefaultEntryForMC, shown only so we can tell the two flags apart:")
print(expression_all["IsDefaultEntryForMC"].value_counts(dropna=False))

expression = expression_all.loc[expression_all["IsDefaultEntryForModel"] == "Yes"].copy()
del expression_all

bookkeeping = [
    "Unnamed: 0",
    "SequencingID",
    "ModelConditionID",
    "IsDefaultEntryForMC",
    "IsDefaultEntryForModel",
]
gene_columns = [column for column in expression.columns if column not in bookkeeping + ["ModelID"]]
expression = expression[["ModelID"] + gene_columns]

print()
print(f"After keeping default profiles: {expression.shape[0]} rows and {expression.shape[1]} columns")
print(f"Unique cell lines: {expression['ModelID'].nunique()}")
print(f"Gene columns: {len(gene_columns)}")
print("First three genes:", gene_columns[:3])
print("Last two genes:", gene_columns[-2:])

duplicate_models = int(expression["ModelID"].duplicated().sum())
if len(expression) == 0:
    print("PROBLEM: the filter kept zero rows. Look at the value counts above.")
elif duplicate_models:
    print(f"PROBLEM: {duplicate_models} cell lines still have more than one default row.")
else:
    print("PASS: one default expression row per cell line.")

example_gene = "TSPAN6 (7105)"
print()
print(f"A sanity check on {example_gene}. These should be small log-expression numbers, not raw counts:")
print(expression[example_gene].describe().round(2))

Values in IsDefaultEntryForModel:
IsDefaultEntryForModel
Yes    2130
No      184
Name: count, dtype: int64

Values in IsDefaultEntryForMC, shown only so we can tell the two flags apart:
IsDefaultEntryForMC
Yes    2234
No       80
Name: count, dtype: int64

After keeping default profiles: 2130 rows and 19219 columns
Unique cell lines: 2130
Gene columns: 19218
First three genes: ['TSPAN6 (7105)', 'TNMD (64102)', 'DPM1 (8813)']
Last two genes: ['F8A2 (474383)', 'F8A1 (8263)']
PASS: one default expression row per cell line.

A sanity check on TSPAN6 (7105). These should be small log-expression numbers, not raw counts:
count    2130.00
mean        3.87
std         1.71
min         0.00
25%         3.38
50%         4.28
75%         4.96
max         8.37
Name: TSPAN6 (7105), dtype: float64


## 5. Drug sensitivity: `GDSC2AUCMatrix.csv`

GDSC2 tested drugs against cell lines and summarized each test as an **AUC** (area under the dose-response curve). In this table, a lower AUC means the drug reduced cell survival more. The cells were more sensitive.

One row is one cell line. One column is one drug. A blank cell means that cell line was not tested with that drug. A blank is not an AUC of zero.

The first column has no name in the file. Its values are `ModelID`s. The other column names are compound codes such as `DPC-006825`, not drug names.

In [5]:
auc = pd.read_csv(data_dir / "GDSC2AUCMatrix.csv")
auc = auc.rename(columns={"Unnamed: 0": "ModelID"})

drug_columns = [column for column in auc.columns if column != "ModelID"]
print(f"AUC matrix: {auc.shape[0]} cell lines and {len(drug_columns)} drug columns")
print(f"Unique ModelIDs: {auc['ModelID'].nunique()}")
print(f"IDs written in DepMap ModelID form: {auc['ModelID'].str.startswith('ACH-').mean():.0%}")
print("First five drug codes:", drug_columns[:5])
print()

missing_fraction = auc[drug_columns].isna().mean()
print(
    "A typical drug has no measurement for "
    f"{missing_fraction.median():.1%} of cell lines."
)
print(f"The most incomplete drug is blank for {missing_fraction.max():.1%} of cell lines.")
print()
print("Corner of the matrix. Numbers are AUCs. Lower means more sensitive:")
auc[["ModelID"] + drug_columns[:4]].head()

AUC matrix: 949 cell lines and 286 drug columns
Unique ModelIDs: 949
IDs written in DepMap ModelID form: 100%
First five drug codes: ['DPC-000001', 'DPC-000002', 'DPC-000027', 'DPC-000028', 'DPC-000034']

A typical drug has no measurement for 7.7% of cell lines.
The most incomplete drug is blank for 76.7% of cell lines.

Corner of the matrix. Numbers are AUCs. Lower means more sensitive:


,ModelID,DPC-000001,DPC-000002,DPC-000027,DPC-000028
0,ACH-000001,0.952757,0.831799,0.885178,0.966006
1,ACH-000002,0.697881,0.421295,0.999851,0.969576
2,ACH-000004,0.758526,0.572811,0.984711,0.998732
3,ACH-000006,0.747942,0.472126,0.990952,NaN
4,ACH-000007,0.778251,0.711866,0.964930,0.979922


## 6. Translate drug codes into names

`GDSC2Log2ViabilityConditions.csv` describes the screen: the compound, the sample that was plated, and the doses that were used. It has many rows per drug because each dose and replicate gets its own row. It does not contain a result for each cell line. The cell-line results are the AUC matrix.

We keep one row per compound and use it as a dictionary:

- `CompoundID`, such as `DPC-006825`, matches the AUC column headers
- `SampleID`, such as `GDSC2:1909`, is the screened sample of that compound
- `CompoundName` is the drug name

In [6]:
conditions = pd.read_csv(data_dir / "GDSC2Log2ViabilityConditions.csv")
print(f"Conditions file: {conditions.shape[0]} rows and {conditions.shape[1]} columns")
print("Columns:", list(conditions.columns))
print()
print("One drug is spread across many rows. Venetoclax is the example:")
venetoclax_rows = conditions.loc[
    conditions["CompoundName"].str.contains("venetoclax", case=False, na=False),
    ["CompoundName", "SampleID", "CompoundID", "Dose", "DoseUnit", "Replicate"],
]
print(f"Venetoclax rows in the conditions file: {len(venetoclax_rows)}")
venetoclax_rows.head(8)

Conditions file: 100557 rows and 7 columns
Columns: ['CompoundName', 'SampleID', 'CompoundID', 'Dose', 'DoseUnit', 'Replicate', 'index']

One drug is spread across many rows. Venetoclax is the example:
Venetoclax rows in the conditions file: 397


,CompoundName,SampleID,CompoundID,Dose,DoseUnit,Replicate
58232,VENETOCLAX,GDSC2:1909,DPC-006825,1.000000,uM,1
58233,VENETOCLAX,GDSC2:1909,DPC-006825,0.316256,uM,1
58234,VENETOCLAX,GDSC2:1909,DPC-006825,0.100018,uM,1
58235,VENETOCLAX,GDSC2:1909,DPC-006825,0.031631,uM,1
58236,VENETOCLAX,GDSC2:1909,DPC-006825,0.010004,uM,1
58237,VENETOCLAX,GDSC2:1909,DPC-006825,0.003164,uM,1
58238,VENETOCLAX,GDSC2:1909,DPC-006825,0.001001,uM,1
58239,VENETOCLAX,GDSC2:1909,DPC-006825,1.000000,uM,2


In [7]:
drug_lookup = (
    conditions[["CompoundID", "SampleID", "CompoundName"]]
    .drop_duplicates()
    .sort_values("CompoundName")
    .reset_index(drop=True)
)

print(f"Unique drugs in the dictionary: {len(drug_lookup)}")
print(f"Unique CompoundIDs: {drug_lookup['CompoundID'].nunique()}")
print(f"Unique SampleIDs: {drug_lookup['SampleID'].nunique()}")
print(f"Unique drug names: {drug_lookup['CompoundName'].nunique()}")

names_per_id = drug_lookup.groupby("CompoundID")["CompoundName"].nunique().max()
ids_per_name = drug_lookup.groupby("CompoundName")["CompoundID"].nunique().max()
print(f"Most names attached to one CompoundID: {names_per_id}")
print(f"Most CompoundIDs attached to one drug name: {ids_per_name}")

compound_ids = set(drug_lookup["CompoundID"])
sample_ids = set(drug_lookup["SampleID"])
matched_compounds = sum(column in compound_ids for column in drug_columns)
matched_samples = sum(column in sample_ids for column in drug_columns)
print()
print(f"AUC columns that match a CompoundID: {matched_compounds} of {len(drug_columns)}")
print(f"AUC columns that match a SampleID: {matched_samples} of {len(drug_columns)}")

if matched_compounds == len(drug_columns) and names_per_id == 1 and ids_per_name == 1:
    print("PASS: every AUC column has exactly one drug name.")
else:
    print("PROBLEM: the drug codes and the names do not line up one to one.")

drug_lookup.head(8)

Unique drugs in the dictionary: 286
Unique CompoundIDs: 286
Unique SampleIDs: 286
Unique drug names: 286
Most names attached to one CompoundID: 1
Most CompoundIDs attached to one drug name: 1

AUC columns that match a CompoundID: 286 of 286
AUC columns that match a SampleID: 0 of 286
PASS: every AUC column has exactly one drug name.


,CompoundID,SampleID,CompoundName
0,DPC-000001,GDSC2:1003,(+)-CAMPTOTHECIN
1,DPC-000002,GDSC2:2172,(+)-JQ-1
2,DPC-000027,GDSC2:1786,1035270-39-3
3,DPC-000028,GDSC2:1625,1044870-39-4
4,DPC-007370,GDSC2:1824,123138
5,DPC-007371,GDSC2:1820,123829
6,DPC-007372,GDSC2:1836,150412
7,DPC-000034,GDSC2:1026,17-AAG


## 7. Is venetoclax in GDSC2?

Venetoclax is the BCL-2 inhibitor for the later case study. This cell looks it up in the dictionary, confirms that its code is a column in the AUC matrix, and counts how many cell lines have a number.

In [8]:
venetoclax = drug_lookup.loc[
    drug_lookup["CompoundName"].str.contains("venetoclax", case=False, na=False)
].copy()

print("Dictionary row for venetoclax:")
display(venetoclax)

if len(venetoclax) == 0:
    print("PROBLEM: venetoclax is not in the conditions file.")
else:
    venetoclax_id = venetoclax["CompoundID"].iloc[0]
    venetoclax_name = venetoclax["CompoundName"].iloc[0]
    in_matrix = venetoclax_id in auc.columns
    measured = int(auc[venetoclax_id].notna().sum()) if in_matrix else 0
    print(f"Drug name in the file: {venetoclax_name}")
    print(f"AUC column exists: {in_matrix}")
    print(f"Cell lines with a venetoclax AUC: {measured}")
    if in_matrix:
        print(f"Lowest AUC: {auc[venetoclax_id].min():.3f}")
        print(f"Highest AUC: {auc[venetoclax_id].max():.3f}")
        print("PASS: venetoclax is in the GDSC2 AUC matrix.")

        peek = (
            auc[["ModelID", venetoclax_id]]
            .dropna()
            .merge(
                models[["ModelID", "CellLineName", "OncotreeLineage"]],
                on="ModelID",
                how="left",
            )
            .rename(columns={venetoclax_id: "venetoclax_auc"})
            .sort_values("venetoclax_auc")
            .head(5)
        )
        print()
        print("Five lowest AUCs. This is only a peek, so we can see that names join correctly:")
        display(peek)

Dictionary row for venetoclax:


,CompoundID,SampleID,CompoundName
272,DPC-006825,GDSC2:1909,VENETOCLAX


Drug name in the file: VENETOCLAX
AUC column exists: True
Cell lines with a venetoclax AUC: 936
Lowest AUC: 0.085
Highest AUC: 1.000
PASS: venetoclax is in the GDSC2 AUC matrix.

Five lowest AUCs. This is only a peek, so we can see that names join correctly:


,ModelID,venetoclax_auc,CellLineName,OncotreeLineage
208,ACH-000363,0.084997,SK-MM-2,Lymphoid
18,ACH-000045,0.118088,MV4-11,Myeloid
893,ACH-002267,0.124753,LC4-1,Lymphoid
29,ACH-000059,0.127550,SUP-B15,Lymphoid
10,ACH-000020,0.171984,MHH-CALL-2,Lymphoid


## 8. Cell lines that appear in more than one file

A biomarker needs both measurements for the same cell line: gene expression, and how that cell line responded to a drug. The cancer type comes from the catalog. The next cell counts those overlaps.

A cell line can be missing from a file for ordinary reasons. It may never have been sequenced, or it may not have been plated in the GDSC2 screen.

In [9]:
expression_ids = set(expression["ModelID"])
auc_ids = set(auc["ModelID"])
catalog_ids = set(models["ModelID"])

shared = expression_ids & auc_ids & catalog_ids
venetoclax_and_expression = expression_ids & set(auc.loc[auc[venetoclax_id].notna(), "ModelID"])

overlap = pd.DataFrame(
    {
        "group": [
            "Cell-line catalog",
            "Default gene-expression profiles",
            "GDSC2 AUC matrix",
            "Expression and AUC",
            "Expression, AUC, and catalog",
            "Venetoclax AUC and expression",
        ],
        "cell_lines": [
            len(catalog_ids),
            len(expression_ids),
            len(auc_ids),
            len(expression_ids & auc_ids),
            len(shared),
            len(venetoclax_and_expression),
        ],
    }
)
display(overlap)

missing_from_catalog = sorted(auc_ids - catalog_ids)
print(f"AUC cell lines missing from the catalog: {len(missing_from_catalog)}")
if missing_from_catalog:
    print("  IDs:", ", ".join(missing_from_catalog))
print(f"AUC cell lines with no default expression profile: {len(auc_ids - expression_ids)}")

shared_models = models.loc[models["ModelID"].isin(shared)]
print(f"Cancer types among cell lines with both expression and AUC: {shared_models['OncotreeLineage'].nunique()}")
print()
print("Largest cancer types in that shared set:")
print(shared_models["OncotreeLineage"].value_counts().head(8).to_string())

,group,cell_lines
0,Cell-line catalog,2554
1,Default gene-expression profiles,2130
2,GDSC2 AUC matrix,949
3,Expression and AUC,708
4,"Expression, AUC, and catalog",708
5,Venetoclax AUC and expression,698


AUC cell lines missing from the catalog: 2
  IDs: ACH-002176, ACH-002194
AUC cell lines with no default expression profile: 241
Cancer types among cell lines with both expression and AUC: 27

Largest cancer types in that shared set:
OncotreeLineage
Lung                    134
Lymphoid                 85
Esophagus/Stomach        51
Breast                   45
Bowel                    43
Skin                     39
CNS/Brain                38
Ovary/Fallopian Tube     34


## 9. What this check decided

When the cells above finish, the files are ready for the biomarker script if all of these are true:

- The default-expression filter leaves one row per cell line.
- Every AUC column matches one drug name.
- Venetoclax has its own column, and hundreds of cell lines have both a venetoclax AUC and expression.
- Hundreds of cell lines sit in both the expression table and the AUC table, with a cancer type in the catalog.

The next step will use those shared cell lines. It will not need the dose-by-dose conditions file again, except as this drug-name dictionary.